<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
</div>

## RapidFire AI + Optuna: Adaptive SFT Hyperparameter Search

This tutorial shows how to use **RFOptuna** for Bayesian hyperparameter optimization
integrated into RapidFire's chunk-based training loop.

**Key difference from RFGridSearch / RFRandomSearch:**
- Grid/Random search decide all configs upfront
- **RFOptuna** uses Optuna's TPE sampler to suggest initial configs, then **prunes underperforming runs after each chunk** and replaces them with smarter suggestions

This notebook uses GPT-2 (124M params) so it runs on any machine.

### Enable Metric Loggers

In [ ]:
import os
os.environ["RF_MLFLOW_ENABLED"] = "true"
os.environ["RF_TENSORBOARD_ENABLED"] = "true"
os.environ["RF_TRACKIO_ENABLED"] = "true"

### Imports

Note: `RFOptuna` requires the `optuna` package. Install with:
```bash
pip install optuna
```

In [ ]:
from rapidfireai import Experiment
from rapidfireai.automl import (
    List,
    Range,
    RFOptuna,
    RFModelConfig,
    RFLoraConfig,
    RFSFTConfig,
)

### Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

train_dataset = dataset["train"].select(range(4))
eval_dataset = dataset["train"].select(range(10, 11))
train_dataset = train_dataset.shuffle(seed=42)
eval_dataset = eval_dataset.shuffle(seed=42)

### Define Data Processing Function

In [ ]:
def sample_formatting_function(example):
    """Format the dataset for GPT-2 while preserving original fields"""
    return {
        "text": f"Question: {example['instruction']}\nAnswer: {example['response']}",
        "instruction": example["instruction"],
        "response": example["response"],
    }

eval_dataset = eval_dataset.map(sample_formatting_function)
train_dataset = train_dataset.map(sample_formatting_function)

### Initialize Experiment

In [ ]:
experiment = Experiment(experiment_name="exp-optuna-chatqa-tiny", mode="fit")

### Define Config Template with Search Space

Instead of listing every combination (GridSearch) or sampling blindly (RandomSearch),
we define a **search space** using `Range(...)` and `List([...])` inside a single
`RFModelConfig`. Optuna's TPE sampler will intelligently explore this space.

**What Optuna controls:**
- `learning_rate`: Sampled from a continuous range
- `lora_alpha`: Sampled from a categorical list

**What stays fixed:**
- Model (`gpt2`), LoRA rank (`r=8`), batch size, etc.

In [ ]:
config_template = RFModelConfig(
    model_name="gpt2",
    peft_config=RFLoraConfig(
        r=8,
        lora_alpha=List([16, 32]),
        lora_dropout=0.1,
        target_modules=["c_attn"],
        bias="none",
    ),
    training_args=RFSFTConfig(
        learning_rate=Range(1e-4, 1e-3),
        lr_scheduler_type="linear",
        per_device_train_batch_size=1,
        max_steps=8,
        logging_steps=2,
        eval_strategy="steps",
        eval_steps=4,
        per_device_eval_batch_size=1,
        fp16=True,
        report_to="none",
    ),
    model_type="causal_lm",
    model_kwargs={
        "device_map": "auto",
        "torch_dtype": "float16",
        "use_cache": False,
    },
    formatting_func=sample_formatting_function,
)

### Create RFOptuna Config Group

Key parameters:
- **`n_initial=4`**: Start with 4 configs (sampled by TPE)
- **`budget=6`**: Allow up to 6 total configs (including replacements for pruned runs)
- **`objective="minimize:eval_loss"`**: Optuna minimizes eval loss to decide pruning
- **`sampler="tpe"`**: Tree-structured Parzen Estimator (Bayesian)
- **`pruner="median"`**: Prune runs performing worse than the median at each chunk

In [ ]:
config_group = RFOptuna(
    configs=[config_template],
    trainer_type="SFT",
    n_initial=4,
    budget=6,
    objective="minimize:eval_loss",
    sampler="tpe",
    pruner="median",
    seed=42,
)

### Define Model Creation Function

In [ ]:
def sample_create_model(model_config):
    """Function to create model object with GPT-2 adjustments"""
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_name = model_config["model_name"]
    model_type = model_config["model_type"]
    model_kwargs = model_config["model_kwargs"]

    if model_type == "causal_lm":
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    else:
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if "gpt2" in model_name.lower():
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "left"
        model.config.pad_token_id = model.config.eos_token_id

    return (model, tokenizer)

### Run Optuna-Powered Training

Behind the scenes:
1. `RFOptuna.get_runs()` asks Optuna's TPE sampler for 4 initial configs
2. RapidFire trains all 4 concurrently using chunk-based scheduling
3. After each chunk completes, the Optuna callback:
   - Reports the run's `eval_loss` to Optuna
   - Checks if Optuna's median pruner wants to prune the run
   - If pruned, suggests a replacement config (up to budget of 6)
4. RapidFire stops pruned runs and starts replacements automatically

In [ ]:
experiment.run_fit(
    config_group,
    sample_create_model,
    train_dataset,
    eval_dataset,
    num_chunks=2,
    seed=42,
)

### Inspect Optuna Study Results

After training, the Optuna study object is accessible on the `RFOptuna` instance.
You can inspect which trials were completed, pruned, or failed.

In [ ]:
study = config_group._study

print(f"Number of trials: {len(study.trials)}")
print(f"Best trial value: {study.best_trial.value:.4f}")
print(f"Best trial params: {study.best_trial.params}")
print()

for trial in study.trials:
    print(f"  Trial {trial.number}: state={trial.state.name}, "
          f"params={trial.params}, value={trial.value}")

### Get Results via RapidFire

In [ ]:
results = experiment.get_results()
results

### End Experiment

In [ ]:
experiment.end()

---

### Comparison: RFOptuna vs RFGridSearch vs RFRandomSearch

| Feature | RFGridSearch | RFRandomSearch | RFOptuna |
|---|---|---|---|
| Config selection | All combos upfront | Random sample upfront | Bayesian (TPE) — learns from results |
| Pruning | Manual via IC Ops | Manual via IC Ops | Automatic (Median / Hyperband pruner) |
| Replacement | Manual clone-modify | Manual clone-modify | Automatic — new suggestions within budget |
| Search space | `List([...])` | `List([...])`, `Range(...)` | `List([...])`, `Range(...)` |
| Best for | Small discrete spaces | Large spaces, no adaptation | Large spaces, adaptive exploration |

<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Thanks for trying RapidFire AI! ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
</div>